SETTING THE LLM

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

groq_llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
)

response = groq_llm.invoke("Say 'LLM is working!' and explain RAG in one sentence.")

print(response.content)


LLM is working! RAG (Retrieval‑Augmented Generation) is a technique that blends a language model’s generative output with relevant documents retrieved from an external knowledge base to improve accuracy and contextual relevance.


Setting up the agents

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent = create_agent(
    model=groq_llm,
    tools=[],
    checkpointer=memory,      # here memory stores the temp data
    system_prompt="You are a helpful assistant.")

agent.invoke({"message": [{"role": "user", "content": "What is RAG?"}]})

SETTING UP PROMPT

In [5]:
promt = " You are a helpful assistant. You are given a question and you have to answer it."

PREPARING PICKLE FILE

* Getting the data 
* EDA
* preprocessing before split
* split
* preprocesing after split
* model

In [12]:
import pandas as pd
df = pd.read_csv('kyphosis.csv')

#Preprocess before split
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Kyphosis'] = le.fit_transform(df['Kyphosis'])

#Split
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Kyphosis'])
y = df['Kyphosis']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

#Preprocess after split
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train[['Age']] = scaler.fit_transform(X_train[['Age']])
X_test[['Age']] = scaler.transform(X_test[['Age']])

#--------------------------------Model Training--------------------------------
# from sklearn.ensemble import RandomForestClassifier
# dt = RandomForestClassifier()
# dt.fit(X_train, y_train)
# preds = dt.predict(X_test)
# from sklearn.metrics import accuracy_score
# accuracy = accuracy_score(y_test, preds) 
# print(accuracy)

In [13]:

from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train,y_train)

preds = dt.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(preds,y_test))

              precision    recall  f1-score   support

           0       0.89      0.77      0.83        22
           1       0.17      0.33      0.22         3

    accuracy                           0.72        25
   macro avg       0.53      0.55      0.53        25
weighted avg       0.81      0.72      0.76        25



# then we dump our pickle file

In [15]:
import pickle

preprocessing = {
    "age_scaler": scaler,
    "label_encoder": le,
}

with open("preprocessing.pkl", "wb") as f:
    pickle.dump(preprocessing, f)

with open("model.pkl", "wb") as f:
    pickle.dump(dt, f)

# testing the pickle file

In [ ]:
model = pickle.load(open("model.pkl", "rb"))
preprocessing = pickle.load(open("preprocessing.pkl", "rb"))

patient = pd.DataFrame({"Age": [15], "Number": [12], "Start": [7]})
patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
prediction = model.predict(patient)
prediction

array([1])

#Creating a tol for our agent for utilizing ml pkl file


In [20]:
def predict_kyphosis(age:int, number:int, start:int)-> int:
    """ This function takes age, number, and start as input and returns the predicted kyphosis class (0 or 1) using the trained model. """
    
    model = pickle.load(open("model.pkl", "rb"))
    preprocessing = pickle.load(open("preprocessing.pkl", "rb"))
    patient = pd.DataFrame({"Age": [age], "Number": [number], "Start": [start]})
    patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
    prediction = model.predict(patient)
    return preprocessing["label_encoder"].inverse_transform(prediction)[0]

# intregating  out kyphosis model into our agent

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

#memory = InMemorySaver()

agent = create_agent(
    model=groq_llm,
    tools=[predict_kyphosis],
   # checkpointer=memory,      # here memory stores the temp data
    system_prompt="""
    You are a helpful assistant.
You are an intelligent Document Question-Answering Assistant with access to the user's private documents through the search_private_documents tool.)""")
agent.invoke({"messages": [{"role": "user", "content": "A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?"}]})

{'messages': [HumanMessage(content='A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?', additional_kwargs={}, response_metadata={}, id='ec8de991-dfe2-4ba3-9db6-03704657fba6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?" They want a prediction of kyphosis class. We need to use the tool predict_kyphosis. We have age=71, number=3, start=5. Use function.', 'tool_calls': [{'id': 'fc_f404d23b-7370-4c81-bf90-02168b8164d3', 'function': {'arguments': '{"age":71,"number":3,"start":5}', 'name': 'predict_kyphosis'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 216, 'total_tokens': 323, 'completion_time': 0.119913908, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.010451016, 'prompt_tokens_details': None, 'queue_time': 0.394848351, 'total_time': 

In [ ]:
patient = pd.DataFrame({"Age": [71], "Number": [3], "Start": [5]})
patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
prediction = model.predict(patient)
return preprocessing["label_encoder"].inverse_transform(prediction)[0]

In [34]:
import os
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


DOCS_FOLDER = os.path.join("Files", "Docs")
INDEX_FOLDER = "files_index"

def load_docs():
    documents = []
    for file_name in os.listdir(DOCS_FOLDER):
        path = os.path.join(DOCS_FOLDER, file_name)

        if file_name.lower().endswith(".pdf"):
            loader = PyPDFLoader(path)



        elif file_name.lower().endswith(".docx"):
            loader = Docx2txtLoader(path)



        elif file_name.lower().endswith((".txt", ".md")):
            loader = TextLoader(path, encoding="utf-8")


        else:
            print("Skipped:", file_name)
            continue

        documents.extend(loader.load())
        print("Loaded doc:", file_name)
    return documents

# load_docs()

#CHUNNKING
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def build_vector_db():
    documents = load_docs()

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(documents)

    vector_db = FAISS.from_documents(chunks, embeddings)
    vector_db.save_local(INDEX_FOLDER)
    print(f"Saved {len(chunks)} chunks to {INDEX_FOLDER}")

build_vector_db()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 876.45it/s]


Loaded doc: Aditya Raj Resume.pdf
Loaded doc: Arman_Resume1.pdf
Saved 8 chunks to files_index


In [37]:
# Retrieval
vector_db = None
def search(question, k=4):
    global vector_db
    if vector_db is None:
        vector_db = FAISS.load_local(INDEX_FOLDER, embeddings, allow_dangerous_deserialization=True)
    return vector_db.similarity_search(question, k=k)
search("python")

[Document(id='ae420e44-037c-435f-94c2-5874a516e951', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-10-13T16:41:03+05:30', 'title': 'Arman - Resume', 'author': 'Arman', 'subject': '(unspecified)', 'moddate': '2026-10-13T16:41:03+05:30', 'source': 'Files\\Docs\\Arman_Resume1.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='ARMAN \nAspiring Data Science & AI/ML Professional \nPhone: 7217358431 | LinkedIn : Arman \nPROFESSIONAL SUMMARY \n \nAspiring Data Science and AI/ML professional with hands -on learning in Python, SQL, Machine Learning \nand Data Science. Familiar with data handling, analysis, visualization and software development tools, with \na strong interest in building practical machine learning solutions and continuously improving technical skills. \nTECHNICAL SKILLS \n \nProgramming Python \nDatabase SQL \nMachine Learning Machine Learning fundamentals \nData Science NumPy, P

In [45]:
# Import the search function from your rag.py file
# from rag import search

def search_private_documents(query: str) -> str:
    """Use this tool to search through the user's private documents, PDFs, text files, and images.
    Always use this tool if the user asks a question about their files, documents, or data.
    Input should be a clear search query."""

    try:
        #RETRIEVAL
        # Call the search function from rag.py
        results = search(query, k=4)

        if not results:
            return "No relevant information found in the documents."

        #AUGMENTATION
        # Format the retrieved chunks into a single string for the LLM to read
        formatted_results = []
        for i, doc in enumerate(results):
            source = doc.metadata.get("source", "Unknown file")
            content = doc.page_content
            formatted_results.append(f"--- Document {i+1} (Source: {source}) ---\n{content}")

        return "\n\n".join(formatted_results)

    except Exception as e:
        return f"An error occurred while searching the documents: {str(e)}"
    
    response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "In the document where arman is studied?"
        }
    ]
})

print(response["messages"][-1].content)

I’m not sure I understand the context of your question. Could you please provide a bit more detail about “the document where Arman studied”? Are you looking for information about a specific publication, a school, a research project, or something else? Once I have a clearer picture, I’ll do my best to help!
